In [ ]:
import os, subprocess, sys

HF_TOKEN = os.environ.get("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
os.environ["PYOPENGL_PLATFORM"] = "egl"

WORK_DIR = os.getcwd()
VIDEO_PATH = os.path.join(WORK_DIR, "video.mp4")
OUTPUT_DIR = os.path.join(WORK_DIR, "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

assert os.path.isfile(VIDEO_PATH), f"Place your video at {VIDEO_PATH}"
print(f"Video found: {VIDEO_PATH}")

In [ ]:
def run(cmd, check=True):
    print(f">>> {cmd[:120]}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout.strip(): print(r.stdout[-1000:])
    if r.returncode != 0:
        if check: print(f"STDERR: {r.stderr[-1000:]}"); raise RuntimeError(f"Failed: {cmd[:80]}")
        else: print(f"(non-fatal) {r.stderr[-300:]}")

run("pip install -q huggingface_hub")
from huggingface_hub import login
login(token=HF_TOKEN)

if not os.path.isdir(os.path.join(WORK_DIR, "sam-3d-body")):
    run("git clone https://github.com/facebookresearch/sam-3d-body.git")

run("apt-get update -qq && apt-get install -y -qq libegl1-mesa-dev libgles2-mesa-dev libgl1-mesa-glx > /dev/null 2>&1 || true", check=False)

run("pip install -q pytorch-lightning pyrender opencv-python yacs scikit-image einops timm "
    "dill pandas rich hydra-core hydra-submitit-launcher hydra-colorlog pyrootutils "
    "webdataset chump 'networkx==3.2.1' roma joblib seaborn wandb appdirs ffmpeg "
    "cython jsonlines xtcocotools loguru optree fvcore pycocotools "
    "tensorboard trimesh pyglet PyOpenGL PyOpenGL-accelerate")

run("pip install -q 'git+https://github.com/facebookresearch/detectron2.git@a1ce2f9' --no-build-isolation --no-deps", check=False)
run("pip install -q git+https://github.com/microsoft/MoGe.git", check=False)

print("Done.")

In [ ]:
SAM3D_REPO = os.path.join(WORK_DIR, "sam-3d-body")
if SAM3D_REPO not in sys.path: sys.path.insert(0, SAM3D_REPO)

import torch, cv2, json, struct
import numpy as np
from tqdm.notebook import tqdm
from sam_3d_body import load_sam_3d_body_hf, SAM3DBodyEstimator

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

model, model_cfg = load_sam_3d_body_hf("facebook/sam-3d-body-dinov3", device=DEVICE)

human_detector = None
try:
    from tools.build_detector import HumanDetector
    human_detector = HumanDetector(name="vitdet", device=DEVICE)
    print("Human detector: loaded")
except Exception as e:
    print(f"Human detector: unavailable ({e})")

fov_estimator = None
try:
    from tools.build_fov_estimator import FOVEstimator
    fov_estimator = FOVEstimator(name="moge2", device=DEVICE)
    print("FOV estimator: loaded")
except Exception as e:
    print(f"FOV estimator: unavailable ({e})")

estimator = SAM3DBodyEstimator(
    sam_3d_body_model=model, model_cfg=model_cfg,
    human_detector=human_detector, human_segmentor=None, fov_estimator=fov_estimator,
)
print("Ready.")

In [ ]:
FRAMES_DIR = os.path.join(OUTPUT_DIR, "frames")
os.makedirs(FRAMES_DIR, exist_ok=True)

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"Video: {width}x{height} @ {fps:.1f} FPS, {total_frames} frames ({total_frames/fps:.1f}s)")

frame_paths = []
idx = 0
while True:
    ret, frame = cap.read()
    if not ret: break
    path = os.path.join(FRAMES_DIR, f"frame_{idx:06d}.jpg")
    cv2.imwrite(path, frame)
    frame_paths.append(path)
    idx += 1
cap.release()

FRAME_SKIP = 2
frame_paths = frame_paths[::FRAME_SKIP]
effective_fps = fps / FRAME_SKIP

MAX_FRAMES = None
if MAX_FRAMES is not None:
    frame_paths = frame_paths[:MAX_FRAMES]
    print(f"** PREVIEW: {MAX_FRAMES} frames **")

print(f"Processing {len(frame_paths)} frames (skip={FRAME_SKIP}, effective {effective_fps:.1f} FPS)")

In [ ]:
all_frame_data = []
faces = None

for frame_idx, frame_path in enumerate(tqdm(frame_paths, desc="Processing")):
    try:
        outputs = estimator.process_one_image(frame_path, bbox_thr=0.5)
    except Exception as e:
        print(f"Frame {frame_idx}: {e}")
        all_frame_data.append([])
        continue

    if faces is None and hasattr(estimator, "faces"):
        faces = estimator.faces

    people = []
    for person in outputs:
        people.append({
            "v": person["pred_vertices"].astype(np.float32),
            "t": person["pred_cam_t"].astype(np.float32),
            "k": person["pred_keypoints_3d"].astype(np.float32),
        })
    all_frame_data.append(people)

print(f"\nDone. {len(all_frame_data)} frames, faces shape: {faces.shape if faces is not None else 'None'}")

In [ ]:
VIEWER_DIR = os.path.join(OUTPUT_DIR, "viewer_data")
os.makedirs(VIEWER_DIR, exist_ok=True)

faces_flat = faces.flatten().astype(np.uint32)
with open(os.path.join(VIEWER_DIR, "faces.bin"), "wb") as f:
    f.write(faces_flat.tobytes())
num_verts = int(faces.max()) + 1
print(f"Faces: {faces.shape[0]} triangles, {num_verts} vertices/person")

CHUNK_SIZE = 60
num_chunks = 0
num_kps = all_frame_data[0][0]["k"].shape[0] if all_frame_data[0] else 70

for chunk_start in range(0, len(all_frame_data), CHUNK_SIZE):
    chunk_frames = all_frame_data[chunk_start : chunk_start + CHUNK_SIZE]
    buf = bytearray()
    for people in chunk_frames:
        buf.append(len(people))
        for p in people:
            buf.extend(p["v"].tobytes())
            buf.extend(p["t"].tobytes())
            buf.extend(p["k"].tobytes())
    with open(os.path.join(VIEWER_DIR, f"chunk_{num_chunks}.bin"), "wb") as f:
        f.write(buf)
    num_chunks += 1

manifest = {
    "totalFrames": len(all_frame_data),
    "fps": effective_fps,
    "chunkSize": CHUNK_SIZE,
    "numChunks": num_chunks,
    "numVerts": num_verts,
    "numKps": num_kps,
}
with open(os.path.join(VIEWER_DIR, "manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2)

total_mb = sum(os.path.getsize(os.path.join(VIEWER_DIR, fn)) for fn in os.listdir(VIEWER_DIR)) / 1024 / 1024
print(f"Exported {num_chunks} binary chunks, total {total_mb:.1f} MB")
print(f"Manifest: {manifest}")

In [ ]:
html_template = r"""
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>SAM 3D Body Viewer</title>
<style>
  * { margin: 0; padding: 0; box-sizing: border-box; }
  body { background: #f5f5f7; color: #1d1d1f; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; overflow: hidden; height: 100vh; display: flex; flex-direction: column; }
  #viewer { flex: 1; position: relative; }
  canvas { display: block; }
  #controls { background: #fff; padding: 10px 16px; display: flex; align-items: center; gap: 10px; border-top: 1px solid #d2d2d7; flex-shrink: 0; }
  #controls button { background: #f5f5f7; border: 1px solid #d2d2d7; color: #1d1d1f; padding: 6px 14px; border-radius: 6px; cursor: pointer; font-size: 13px; transition: background 0.15s; }
  #controls button:hover { background: #e8e8ed; }
  #controls button.active { background: #0071e3; color: #fff; border-color: #0071e3; }
  #timeline { flex: 1; -webkit-appearance: none; appearance: none; height: 5px; background: #d2d2d7; border-radius: 3px; outline: none; cursor: pointer; }
  #timeline::-webkit-slider-thumb { -webkit-appearance: none; width: 14px; height: 14px; border-radius: 50%; background: #0071e3; cursor: grab; }
  #frame-info { font-size: 12px; color: #86868b; min-width: 130px; text-align: right; font-variant-numeric: tabular-nums; }
  #loading { position: absolute; top: 50%; left: 50%; transform: translate(-50%,-50%); font-size: 18px; color: #0071e3; z-index: 100; }
  #speed-label { font-size: 11px; color: #86868b; }
  #return-indicator { position: absolute; bottom: 60px; left: 50%; transform: translateX(-50%); background: rgba(0,0,0,0.55); color: #fff; padding: 5px 14px; border-radius: 16px; font-size: 11px; z-index: 20; opacity: 0; pointer-events: none; transition: opacity 0.4s; }
  #return-indicator.visible { opacity: 1; }
  #people-panel { position: absolute; top: 10px; right: 10px; background: rgba(255,255,255,0.92); border-radius: 10px; padding: 10px; box-shadow: 0 2px 12px rgba(0,0,0,0.1); z-index: 15; min-width: 140px; backdrop-filter: blur(8px); }
  #people-panel h3 { font-size: 12px; color: #86868b; margin-bottom: 6px; font-weight: 600; text-transform: uppercase; letter-spacing: 0.5px; }
  .person-btn { display: block; width: 100%; text-align: left; background: #f0f0f5; border: 1px solid #d2d2d7; color: #1d1d1f; padding: 6px 10px; border-radius: 6px; cursor: pointer; font-size: 13px; margin-bottom: 4px; transition: background 0.15s; }
  .person-btn:hover { background: #e0e0ea; }
  .person-btn.selected { background: #0071e3; color: #fff; border-color: #0071e3; }
  .person-btn.all-btn { font-weight: 600; }
</style>
</head>
<body>
<div id=\"viewer\">
  <div id=\"loading\">Loading...</div>
  <div id=\"return-indicator\">Returning to default view...</div>
  <div id=\"people-panel\"><h3>People</h3><button class=\"person-btn all-btn selected\" data-pid=\"-1\">All</button></div>
</div>
<div id=\"controls\">
  <button id=\"btn-play\">Play</button>
  <button id=\"btn-prev\">&larr;</button>
  <button id=\"btn-next\">&rarr;</button>
  <input type=\"range\" id=\"timeline\" min=\"0\" max=\"0\" value=\"0\">
  <span id=\"frame-info\">0 / 0</span>
  <button id=\"btn-speed\">1x</button>
  <span id=\"speed-label\">speed</span>
  <button id=\"btn-wireframe\">Wire</button>
  <button id=\"btn-bones\">Bones</button>
</div>

<script type=\"importmap\">\n{ \"imports\": { \"three\": \"https://cdn.jsdelivr.net/npm/three@0.170.0/build/three.module.js\", \"three/addons/\": \"https://cdn.jsdelivr.net/npm/three@0.170.0/examples/jsm/\" } }\n</script>

<script type=\"module\">
import * as THREE from 'three';
import { OrbitControls } from 'three/addons/controls/OrbitControls.js';

const D = '__DATA_PATH__';
const M = await fetch(D+'/manifest.json').then(r=>r.json());
const faceBuf = await fetch(D+'/faces.bin').then(r=>r.arrayBuffer());
const FACES = new Uint32Array(faceBuf);
const {totalFrames,fps:VIDEO_FPS,chunkSize:CS,numChunks:NC,numVerts:NV,numKps:NK} = M;

document.getElementById('loading').textContent='Loading first chunk...';

const cache=new Map(), loading=new Map();
const BYTES_PER_PERSON = NV*3*4 + 3*4 + NK*3*4;

async function loadChunk(ci){
  if(cache.has(ci)) return cache.get(ci);
  if(loading.has(ci)) return loading.get(ci);
  const p=fetch(D+'/chunk_'+ci+'.bin').then(r=>r.arrayBuffer()).then(ab=>{
    const dv=new DataView(ab);
    const frames=[]; let off=0;
    while(off<ab.byteLength){
      const np=dv.getUint8(off); off++;
      const people=[];
      for(let i=0;i<np;i++){
        const v=new Float32Array(ab,off,NV*3); off+=NV*3*4;
        const t=new Float32Array(ab,off,3); off+=12;
        const k=new Float32Array(ab,off,NK*3); off+=NK*3*4;
        people.push({v,t,k});
      }
      frames.push(people);
    }
    cache.set(ci,frames); loading.delete(ci);
    return frames;
  });
  loading.set(ci,p); return p;
}
function getFrame(fi){ const c=cache.get(Math.floor(fi/CS)); return c?c[fi%CS]:null; }
function prefetch(fi){ const ci=Math.floor(fi/CS); for(let d=0;d<=2;d++) if(ci+d<NC) loadChunk(ci+d); }

await loadChunk(0);
document.getElementById('loading').remove();

const BONES=[[13,11],[11,9],[14,12],[12,10],[9,10],[5,9],[6,10],[5,6],[5,7],[6,8],[7,62],[8,41],[1,2],[0,1],[0,2],[1,3],[2,4],[3,5],[4,6],[13,15],[13,16],[13,17],[14,18],[14,19],[14,20]];

const container=document.getElementById('viewer');
const scene=new THREE.Scene(); scene.background=new THREE.Color(0xffffff);
const camera=new THREE.PerspectiveCamera(50,container.clientWidth/container.clientHeight,0.1,1000);
camera.position.set(0,0,5);
const ren=new THREE.WebGLRenderer({antialias:true});
ren.setSize(container.clientWidth,container.clientHeight);
ren.setPixelRatio(Math.min(devicePixelRatio,2));
ren.outputColorSpace=THREE.SRGBColorSpace;
container.appendChild(ren.domElement);
const ctrl=new OrbitControls(camera,ren.domElement); ctrl.enableDamping=true; ctrl.dampingFactor=0.08;

scene.add(new THREE.AmbientLight(0xffffff,0.3));
for(const phi of[0,2*Math.PI/3,4*Math.PI/3]){const th=Math.PI/6;const l=new THREE.DirectionalLight(0xffffff,1.0);l.position.set(Math.sin(th)*Math.cos(phi)*10,Math.sin(th)*Math.sin(phi)*10,Math.cos(th)*10);scene.add(l);}

const idxArr=new Uint32Array(FACES);
const MC=new THREE.Color(0.651,0.741,0.859), DC=new THREE.Color(0.85,0.87,0.90);
const ms=c=>new THREE.MeshStandardMaterial({color:c,roughness:1,metalness:0,side:THREE.DoubleSide});
const mw=c=>new THREE.MeshStandardMaterial({color:c,roughness:1,metalness:0,wireframe:true,side:THREE.DoubleSide});
const matS=ms(MC),matW=mw(MC),matDS=ms(DC),matDW=mw(DC);
const matB=new THREE.MeshStandardMaterial({color:new THREE.Color(0.9,0.25,0.3),roughness:0.6,metalness:0});
const matJ=new THREE.MeshStandardMaterial({color:new THREE.Color(0.95,0.35,0.2),roughness:0.4,metalness:0});
const jGeo=new THREE.SphereGeometry(0.008,8,6), bGeo=new THREE.CylinderGeometry(0.004,0.004,1,6);

let wire=true, bones=true, selP=-1;
let pms=[];
const ray=new THREE.Raycaster(), mouse=new THREE.Vector2();

function tkp(k,i,t){return new THREE.Vector3(k[i*3]+t[0],-(k[i*3+1]+t[1]),-(k[i*3+2]+t[2]));}

function mkBones(k,t){
  const g=new THREE.Group();
  for(let j=0;j<Math.min(NK,21);j++){const s=new THREE.Mesh(jGeo,matJ);s.position.copy(tkp(k,j,t));g.add(s);}
  for(const[a,b]of BONES){
    if(a>=NK||b>=NK)continue;
    const pa=tkp(k,a,t),pb=tkp(k,b,t),dir=new THREE.Vector3().subVectors(pb,pa),len=dir.length();
    if(len<0.001)continue;
    const c=new THREE.Mesh(bGeo,matB);c.scale.set(1,len,1);
    c.position.copy(pa).add(pb).multiplyScalar(0.5);
    c.quaternion.setFromUnitVectors(new THREE.Vector3(0,1,0),dir.normalize());
    g.add(c);
  }
  g.visible=bones; return g;
}

let camInit=false;
function setFrame(fi){
  const ppl=getFrame(fi); if(!ppl)return; prefetch(fi);
  while(pms.length>ppl.length){const o=pms.pop();scene.remove(o.m);o.m.geometry.dispose();scene.remove(o.b);o.b.traverse(c=>{if(c.geometry)c.geometry.dispose();});}
  updatePanel(ppl.length);
  for(let i=0;i<ppl.length;i++){
    const p=ppl[i],V=NV;
    const pos=new Float32Array(V*3);
    for(let vi=0;vi<V;vi++){pos[vi*3]=p.v[vi*3]+p.t[0];pos[vi*3+1]=-(p.v[vi*3+1]+p.t[1]);pos[vi*3+2]=-(p.v[vi*3+2]+p.t[2]);}
    const sel=selP===-1||selP===i;
    const mat=wire?(sel?matW:matDW):(sel?matS:matDS);
    if(i<pms.length){
      const g=pms[i].m.geometry;g.setAttribute('position',new THREE.BufferAttribute(pos,3));g.computeVertexNormals();g.computeBoundingSphere();
      pms[i].m.material=mat;pms[i].m.userData.pid=i;
      scene.remove(pms[i].b);pms[i].b.traverse(c=>{if(c.geometry)c.geometry.dispose();});
      const bg=mkBones(p.k,p.t);bg.visible=bones&&sel;scene.add(bg);pms[i].b=bg;
    }else{
      const g=new THREE.BufferGeometry();g.setAttribute('position',new THREE.BufferAttribute(pos,3));g.setIndex(new THREE.BufferAttribute(idxArr,1));g.computeVertexNormals();
      const m=new THREE.Mesh(g,mat);m.userData.pid=i;scene.add(m);
      const bg=mkBones(p.k,p.t);bg.visible=bones&&sel;scene.add(bg);
      pms.push({m,b:bg});
    }
  }
  if(ppl.length>0&&!camInit){
    const p=ppl[0];ctrl.target.set(p.t[0],-p.t[1],-p.t[2]);camera.position.set(p.t[0],-p.t[1],-p.t[2]+4);
    ctrl.update();saveDef();camInit=true;
  }
}

const panel=document.getElementById('people-panel');
let lastPC=0;
function updatePanel(n){
  if(n===lastPC)return;lastPC=n;
  panel.innerHTML='<h3>People</h3>';
  const a=document.createElement('button');a.className='person-btn all-btn'+(selP===-1?' selected':'');a.textContent='All';a.dataset.pid=-1;a.onclick=()=>selPerson(-1);panel.appendChild(a);
  for(let i=0;i<n;i++){const b=document.createElement('button');b.className='person-btn'+(selP===i?' selected':'');b.textContent='Person '+(i+1);b.dataset.pid=i;b.onclick=()=>selPerson(i);panel.appendChild(b);}
}
function selPerson(pid){
  selP=pid;panel.querySelectorAll('.person-btn').forEach(b=>b.classList.toggle('selected',parseInt(b.dataset.pid)===pid));
  if(pid>=0&&pid<pms.length){const bb=new THREE.Box3().setFromObject(pms[pid].m);const c=bb.getCenter(new THREE.Vector3());const s=bb.getSize(new THREE.Vector3()).length();ctrl.target.copy(c);camera.position.set(c.x,c.y,c.z+s*1.2);ctrl.update();}
  for(let i=0;i<pms.length;i++){const sel=selP===-1||selP===i;pms[i].m.material=wire?(sel?matW:matDW):(sel?matS:matDS);pms[i].b.visible=bones&&sel;}
}
ren.domElement.addEventListener('click',e=>{const r2=ren.domElement.getBoundingClientRect();mouse.x=((e.clientX-r2.left)/r2.width)*2-1;mouse.y=-((e.clientY-r2.top)/r2.height)*2+1;ray.setFromCamera(mouse,camera);const h=ray.intersectObjects(pms.map(p=>p.m));if(h.length>0)selPerson(h[0].object.userData.pid);});

let curF=0,playing=false,spd=1,acc=0;
const spds=[0.25,0.5,1,2,4];let si=2;
const tl=document.getElementById('timeline'),fi2=document.getElementById('frame-info');
const bp=document.getElementById('btn-play'),bpr=document.getElementById('btn-prev'),bn=document.getElementById('btn-next');
const bs=document.getElementById('btn-speed'),bw=document.getElementById('btn-wireframe'),bb=document.getElementById('btn-bones');
const ri=document.getElementById('return-indicator');
tl.max=totalFrames-1;

let defP=new THREE.Vector3(0,0,5),defT=new THREE.Vector3(0,0,0),uiAt=0,retDef=false;
function saveDef(){defP.copy(camera.position);defT.copy(ctrl.target);}
function startRet(){uiAt=performance.now();retDef=false;ri.classList.remove('visible');}
ctrl.addEventListener('start',startRet);ctrl.addEventListener('change',()=>{uiAt=performance.now();});

function upUI(){tl.value=curF;const s=(curF/VIDEO_FPS).toFixed(1),ts=(totalFrames/VIDEO_FPS).toFixed(1);fi2.textContent=`${curF}/${totalFrames-1} (${s}s/${ts}s)`;bp.textContent=playing?'Pause':'Play';bp.classList.toggle('active',playing);}
async function goF(f){curF=Math.max(0,Math.min(totalFrames-1,f));await loadChunk(Math.floor(curF/CS));setFrame(curF);upUI();}

bp.addEventListener('click',()=>{playing=!playing;acc=0;upUI();});
bpr.addEventListener('click',()=>{playing=false;goF(curF-1);});
bn.addEventListener('click',()=>{playing=false;goF(curF+1);});
tl.addEventListener('input',()=>{curF=Math.max(0,Math.min(totalFrames-1,parseInt(tl.value)));acc=0;loadChunk(Math.floor(curF/CS)).then(()=>{setFrame(curF);upUI();});startRet();});
bs.addEventListener('click',()=>{si=(si+1)%spds.length;spd=spds[si];bs.textContent=spd+'x';});
bw.addEventListener('click',()=>{wire=!wire;bw.textContent=wire?'Wire':'Solid';pms.forEach((o,i)=>{const sel=selP===-1||selP===i;o.m.material=wire?(sel?matW:matDW):(sel?matS:matDS);});});
bb.addEventListener('click',()=>{bones=!bones;bb.classList.toggle('active',bones);pms.forEach((o,i)=>{const sel=selP===-1||selP===i;o.b.visible=bones&&sel;});});
bb.classList.add('active');

document.addEventListener('keydown',e=>{
  if(e.code==='Space'){e.preventDefault();playing=!playing;acc=0;upUI();}
  if(e.code==='ArrowLeft'){playing=false;goF(curF-1);}
  if(e.code==='ArrowRight'){playing=false;goF(curF+1);}
  if(e.code==='KeyW')bw.click();
  if(e.code==='KeyB')bb.click();
  if(e.code==='Escape')selPerson(-1);
});

let lt=0;const fd=1000/VIDEO_FPS;
function anim(t){
  requestAnimationFrame(anim);const dt=t-lt;lt=t;
  if(playing){acc+=dt*spd;while(acc>=fd){acc-=fd;curF++;if(curF>=totalFrames)curF=0;}const ci=Math.floor(curF/CS);if(cache.has(ci)){setFrame(curF);upUI();}else loadChunk(ci);}
  if(uiAt>0){const el=performance.now()-uiAt;const d=camera.position.distanceTo(defP)+ctrl.target.distanceTo(defT);if(el>4000&&d>0.01){retDef=true;ri.classList.add('visible');camera.position.lerp(defP,0.03);ctrl.target.lerp(defT,0.03);}else if(d<=0.01&&retDef){retDef=false;ri.classList.remove('visible');uiAt=0;camera.position.copy(defP);ctrl.target.copy(defT);}}
  ctrl.update();ren.render(scene,camera);
}
goF(0);requestAnimationFrame(anim);
window.addEventListener('resize',()=>{camera.aspect=container.clientWidth/container.clientHeight;camera.updateProjectionMatrix();ren.setSize(container.clientWidth,container.clientHeight);});
</script>
</body></html>
"""

viewer_data_rel = os.path.relpath(VIEWER_DIR, OUTPUT_DIR)
html_out = html_template.replace('__DATA_PATH__', viewer_data_rel)
viewer_path = os.path.join(OUTPUT_DIR, "sam3d_viewer.html")
with open(viewer_path, "w") as f:
    f.write(html_out)

with open(os.path.join(OUTPUT_DIR, "serve.py"), "w") as f:
    f.write('#!/usr/bin/env python3\nimport http.server,socketserver,os,webbrowser\nPORT=8080\nos.chdir(os.path.dirname(os.path.abspath(__file__)))\nclass H(http.server.SimpleHTTPRequestHandler):\n  def end_headers(self):\n    self.send_header("Access-Control-Allow-Origin","*");super().end_headers()\nprint(f"http://localhost:{PORT}/sam3d_viewer.html");webbrowser.open(f"http://localhost:{PORT}/sam3d_viewer.html")\nwith socketserver.TCPServer(("",PORT),H) as s: s.serve_forever()\n')

print(f"Viewer: {viewer_path} ({os.path.getsize(viewer_path)/1024:.0f} KB)")
print(f"Data:   {VIEWER_DIR}/ ({total_mb:.1f} MB)")
print(f"Test:   cd output && python serve.py")
print(f"\nDone! Download output/ folder, upload to any web host, iframe sam3d_viewer.html")